In [ ]:
import tskit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, SVG
from scipy.linalg import expm

In [ ]:
tables = tskit.TableCollection(sequence_length=1)

# Nodes 0–4 are samples; nodes 5–8 are ancestors.
for u, time in enumerate([0, 0, 0, 0, 0, 1, 1, 2, 3]):
    tables.nodes.add_row(
        flags=tskit.NODE_IS_SAMPLE if u < 5 else 0,
        time=time,
    )

for parent, child in [
    (5, 0), (5, 1),
    (6, 3), (6, 4),
    (7, 2), (7, 6),
    (8, 5), (8, 7),
]:
    tables.edges.add_row(left=0, right=1, parent=parent, child=child)

# One site, assuming ancestral G.
site = tables.sites.add_row(position=0.5, ancestral_state="G")
for node, allele, time in [
    (0, "C", 0.5),
    (1, "A", 0.5),
    (2, "C", 1.0),
    (3, "A", 0.5),
]:
    tables.mutations.add_row(
        site=site, node=node, derived_state=allele, time=time
    )

tables.sort()
ts = tables.tree_sequence()
tree = ts.first()

labels = {u: str(u) for u in range(9)}
labels.update({0: "0: C", 1: "1: A", 2: "2: C", 3: "3: A", 4: "4: G"})

display(SVG(tree.draw_svg(
    size=(740, 440),
    time_scale="time",
    node_labels=labels,
    mutation_labels={m.id: str(m.id) for m in ts.mutations()},
    style=".mut .sym {stroke: red;} .mut .lab {fill: red;}",
)))

In [ ]:
def calculate_prob_matrix(mu, num_states = 4):
    """Calculate the transition matrix for a Jukes-Cantor model."""
    Q = np.full((num_states, num_states), mu)
    np.fill_diagonal(Q, -(num_states - 1) * mu)
    return Q

def transition_matrix(Q, t):
    return expm(Q * t)

In [ ]:
def calculate_jukes_cantor_prob(t_branch):
    """Calculate the transition matrix for a Jukes-Cantor model."""
    p_same = 1/4 + 3/4 * np.exp(-4*t_branch/3)
    p_diff = 1/4 - 1/4 * np.exp(-4*t_branch/3)

    P = np.full((4, 4), p_diff)
    np.fill_diagonal(P, p_same)

    return P

In [ ]:
# Rows = from, columns = to; order: A, C, G, T
bases = ["A", "C", "G", "T"]
base_index = {base: i for i, base in enumerate(bases)}

In [ ]:
def felsenstein_backwards_jc(tree, tip_states):
    "felsenstein algorithm to calculate the likelihood of the tree given the tip states"

    #likelihoods = {}  # Dictionary to store likelihoods for each node
    likelihoods = {u: np.zeros(4) for u in tree.nodes()}  # Initialize likelihoods for all nodes
    best = {u: np.zeros(4) for u in tree.nodes()}  # joint: probability of the best assignment below each node
    choice = {}  # joint: choice[child][i] = best state for the child if its parent is in state i
    message = {}  # marginal: message[child] = P @ likelihoods[child], what the child tells its parent

    # set likelihoods for the tips
    for u in tree.nodes():
        if tree.is_leaf(u):
            state = tip_states[u]
            likelihoods[u] = np.zeros(4)
            likelihoods[u][base_index[state]] = 1.0
            best[u] = likelihoods[u].copy()
        else:
            likelihoods[u] = np.ones(4)  # Initialize internal nodes with uniform likelihoods, 1 since we will multiply in the children's contributions
            best[u] = np.ones(4)

    for u in tree.nodes(order="postorder"):
            if tree.is_internal(u):
                # start this node's costs at 0 for every state, then add each child's contribution
                # print(u)
                #node_state_costs[u] = np.zeros(len(bases))
                children = tree.children(u)
                
                for child in children:
                    branch_length = tree.branch_length(child)
                    # later we can upgrade this so that we can save the transition matrix for each branch length, but for now we will just calculate it on the fly
                    P = calculate_jukes_cantor_prob(branch_length)
                    # sum over all possible states of the child, weighted by the transition probabilities
                    child_states_sum = P @ likelihoods[child]
                    # multiply the parent's likelihoods by the sum of the child's states
                    likelihoods[u] *= child_states_sum
                    # print("likelihoods:", likelihoods[u])

                    # marginal
                    message[child] = child_states_sum      # P @ likelihoods[child]

                    # joint: same products, but keep the largest
                    # scores[i, j] = P[i, j] * best[child][j]   (parent in state i, child in state j)
                    scores = P * best[child]
                    # for each parent state i, remember which child state j was best
                    choice[child] = scores.argmax(axis=1)
                    # multiply the parent's best by the child's best score
                    best[u] *= scores.max(axis=1)

    # joint: the root picks its own state (JC69: every base has frequency 1/4)
    root = tree.root
    root_scores = 0.25 * best[root]
    path_probability = root_scores.max()

    # joint: traceback from the root to the leaves, reading off the stored choices
    states = {root: bases[root_scores.argmax()]}
    for u in tree.nodes(order="preorder"):
        if u != root:
            parent_index = base_index[states[tree.parent(u)]]
            states[u] = bases[choice[u][parent_index]]

    # marginal: how probable is each state at each node
    outside = {tree.root: np.full(4, 0.25)}
    marginal = {}

    for u in tree.nodes(order="preorder"):
        if u != tree.root:
            parent = tree.parent(u)
            # everything the parent knows, except what came from u
            rest = outside[parent].copy()
            for sibling in tree.children(parent):
                if sibling != u:
                    rest *= message[sibling]
            # carry it down the branch to u: sum over the parent's base
            P = calculate_jukes_cantor_prob(tree.branch_length(u))
            outside[u] = rest @ P

        marginal[u] = likelihoods[u] * outside[u]
        marginal[u] = marginal[u] / marginal[u].sum()
            
    return likelihoods, states, path_probability, marginal


In [ ]:
likelihoods, states, path_probability, marginal = felsenstein_backwards_jc(tree, tip_states={0: "C", 1: "A", 2: "C", 3: "A", 4: "G"})


In [ ]:

print("Likelihoods at each node:"   )
for u in tree.nodes():
    print(f"Node {u}: {likelihoods[u]}")
print(f"Most likely states at each node: {states}")
print(f"Path probability: {path_probability}")

print("Marginal probabilities at each node:")
for u in tree.nodes():
    print(f"Node {u}: {marginal[u]}")
marginal_states = {u: bases[marginal[u].argmax()] for u in tree.nodes()}
print(f"Marginal states at each node: {marginal_states}")

In [ ]:
display(SVG(tree.draw_svg(
    size=(740, 440),
    time_scale="time",
    node_labels=labels,
    mutation_labels={m.id: str(m.id) for m in ts.mutations()},
    style=".mut .sym {stroke: red;} .mut .lab {fill: red;}",
)))

In [ ]:
from make_and_plot_random_tree import random_tree, print_branch_lengths, save_tree_plot, plot_tree

import matplotlib.pyplot as plt

tree = random_tree(5, equal_branch_lengths=False, branch_length=0.2, seed=2)
observed_bases = {0: "A", 1: "A", 2: "G", 3: "A", 4: "G"}

# Or show it on screen (in a Jupyter notebook it appears under the cell)
plot_tree(tree, title="My tree")
plt.show()


In [ ]:
likelihoods, states, path_probability, marginal = felsenstein_backwards_jc(tree, tip_states=observed_bases)
print("Likelihoods at each node:"   )
for u in tree.nodes():
    print(f"Node {u}: {likelihoods[u]}")
print(f"Most likely states at each node: {states}")
print(f"Path probability: {path_probability}")
marginal_states = {u: bases[marginal[u].argmax()] for u in tree.nodes()}
print(f"Marginal states at each node: {marginal_states}")

In [ ]:
print